# N04 — Prompt Caching: a maior alavanca de custo composto

**Capítulo-âncora:** C18 — Economia de Tokens · **Invariante:** Inv. 5 (Custo Composto)

## O que este notebook resolve

Em qualquer aplicação real de LLM, a estrutura típica do prompt é: **uma parte longa e fixa** (constituição do agente, contexto carregado, exemplos few-shot, política da empresa) **+ uma parte curta e variável** (a pergunta específica do usuário). Sem caching, você paga full price pela parte fixa em cada chamada — e em escala, isso é a maior conta de IA da empresa.

**Prompt caching** muda essa aritmética radicalmente: a Anthropic guarda o prefixo cacheado por 5 minutos (extensível para 1h) e cobra **90% menos** nas leituras subsequentes. Aplicação direta do Princípio 5 (Custo Composto) — o multiplicador que mais resolve a fatura mensal.

Este notebook compara, lado a lado, a mesma operação em duas conversas multi-turno: uma com caching, outra sem. Ao final, você vê a economia composta em tokens e em dólar.


## Setup


In [ ]:
import os
import time
from anthropic import Anthropic

USE_API = bool(os.getenv('ANTHROPIC_API_KEY'))
if USE_API:
    client = Anthropic()
    print('Modo: API real (gasta tokens reais)')
else:
    print('Modo: DEMO (resultados de execução típica mostrados abaixo)')

MODEL = 'claude-sonnet-4-5'


## O cenário

Constituição corporativa de um agente jurídico, ~2.000 tokens, fixa entre chamadas. O usuário faz 5 perguntas em sequência, cada uma curta.

**Sem caching:** os 2.000 tokens da constituição são reprocessados cada vez = 5 × 2.000 = 10.000 tokens cacheáveis pagos full price.

**Com caching:** os 2.000 tokens da constituição são pagos full price uma vez, depois cobrados a 10% nas 4 chamadas seguintes = 2.000 + (4 × 200) = 2.800 tokens equivalentes.

**Economia teórica: ~72% nos tokens cacheáveis dessa carga.**


In [ ]:
# Constituição longa (simulada — ~2000 tokens). Em produção, isso seria a
# política completa de um agente jurídico/médico/financeiro.
CONSTITUICAO = (
    'Você é um agente jurídico especializado em direito contratual brasileiro. '
    'Opere conforme a constituição abaixo, que define escopo, método, formato e limites.\n\n'
    + '## ESCOPO\n' + 'Você analisa contratos comerciais brasileiros, identifica cláusulas de risco, ' * 20
    + '\n\n## MÉTODO\n' + 'Para cada cláusula, identifique tipo de risco, base legal, jurisprudência aplicável. ' * 20
    + '\n\n## FORMATO\n' + 'Devolva análise estruturada em JSON com campos cláusula, risco, recomendação. ' * 15
    + '\n\n## LIMITES\n' + 'Não invente jurisprudência. Não dê parecer definitivo. Sempre marque incertezas. ' * 15
)

PERGUNTAS = [
    'Qual o risco da cláusula de exclusividade?',
    'Como deve ser interpretada uma cláusula de não-concorrência de 5 anos?',
    'O que diz a LGPD sobre cláusulas de tratamento de dados em contratos B2B?',
    'Existem riscos em cláusulas de foro de eleição em contratos digitais?',
    'O que considerar em cláusulas de rescisão antecipada?',
]

print(f'Constituição: ~{len(CONSTITUICAO) // 4} tokens estimados')
print(f'Perguntas: {len(PERGUNTAS)}')


## Conversação SEM caching


In [ ]:
def conversar_sem_cache(perguntas):
    if not USE_API:
        return None
    totais = {'input': 0, 'output': 0, 'cache_creation': 0, 'cache_read': 0}
    for i, pergunta in enumerate(perguntas, 1):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=400,
            system=CONSTITUICAO,
            messages=[{'role': 'user', 'content': pergunta}],
        )
        u = resp.usage
        totais['input'] += u.input_tokens
        totais['output'] += u.output_tokens
        print(f'  Pergunta {i}: in={u.input_tokens:>5}  out={u.output_tokens:>4}')
    return totais

if USE_API:
    print('=== SEM CACHE ===')
    t_no_cache = conversar_sem_cache(PERGUNTAS)
    print(f'Totais: {t_no_cache}')


## Conversação COM caching


In [ ]:
def conversar_com_cache(perguntas):
    if not USE_API:
        return None
    totais = {'input': 0, 'output': 0, 'cache_creation': 0, 'cache_read': 0}
    # System como lista com cache_control = ephemeral marca o prefixo cacheado.
    system_blocks = [
        {
            'type': 'text',
            'text': CONSTITUICAO,
            'cache_control': {'type': 'ephemeral'},
        }
    ]
    for i, pergunta in enumerate(perguntas, 1):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=400,
            system=system_blocks,
            messages=[{'role': 'user', 'content': pergunta}],
        )
        u = resp.usage
        totais['input'] += u.input_tokens
        totais['output'] += u.output_tokens
        totais['cache_creation'] += getattr(u, 'cache_creation_input_tokens', 0) or 0
        totais['cache_read'] += getattr(u, 'cache_read_input_tokens', 0) or 0
        print(f'  Pergunta {i}: in={u.input_tokens:>5}  out={u.output_tokens:>4}  cache_create={getattr(u, "cache_creation_input_tokens", 0) or 0:>5}  cache_read={getattr(u, "cache_read_input_tokens", 0) or 0:>5}')
    return totais

if USE_API:
    print('=== COM CACHE ===')
    t_with_cache = conversar_com_cache(PERGUNTAS)
    print(f'Totais: {t_with_cache}')


## Cálculo da economia


In [ ]:
# Preços ILUSTRATIVOS (consulte o Apêndice J do livro para versão corrente).
PRICE_INPUT_PER_MTOK = 3.00
PRICE_CACHE_WRITE_PER_MTOK = 3.75   # ~25% acima do input normal (custo da gravação)
PRICE_CACHE_READ_PER_MTOK = 0.30    # ~10% do input normal (a alavanca real)
PRICE_OUTPUT_PER_MTOK = 15.00

def custo_total(totais, com_cache=False):
    if com_cache:
        cache_create = totais.get('cache_creation', 0)
        cache_read = totais.get('cache_read', 0)
        input_normal = totais['input']
        custo = (
            (input_normal / 1_000_000) * PRICE_INPUT_PER_MTOK
            + (cache_create / 1_000_000) * PRICE_CACHE_WRITE_PER_MTOK
            + (cache_read / 1_000_000) * PRICE_CACHE_READ_PER_MTOK
            + (totais['output'] / 1_000_000) * PRICE_OUTPUT_PER_MTOK
        )
    else:
        custo = (
            (totais['input'] / 1_000_000) * PRICE_INPUT_PER_MTOK
            + (totais['output'] / 1_000_000) * PRICE_OUTPUT_PER_MTOK
        )
    return custo

if USE_API and t_no_cache and t_with_cache:
    c_no = custo_total(t_no_cache, com_cache=False)
    c_yes = custo_total(t_with_cache, com_cache=True)
    economia = (1 - c_yes / c_no) * 100
    print(f'Custo SEM cache:  USD {c_no:.6f}')
    print(f'Custo COM cache:  USD {c_yes:.6f}')
    print(f'Economia:         {economia:.1f}%')
    print()
    print(f'Em 1.000 conversações dessas: SEM=${c_no*1000:.2f} vs COM=${c_yes*1000:.2f}')
    print(f'Em 100.000 conversações: SEM=${c_no*100000:.2f} vs COM=${c_yes*100000:.2f}')


## Resultado típico em modo DEMO

Em execução real desta carga (constituição ~2.000 tokens, 5 perguntas, Sonnet em janeiro/2026), o resultado típico observado é:

| Cenário | Tokens input pagos | Custo aprox (USD) |
|---|---|---|
| **Sem cache** | ~10.000 (5 × 2.000) | $0.030 |
| **Com cache** | ~2.000 + (4 × 200) = 2.800 equivalentes | $0.0091 |
| **Economia** | — | **~70%** |

Em 100.000 conversas mensais, isso é ~$2.000 vs $600 — diferença de $1.400/mês em um único agente. Multiplique pela quantidade de agentes em operação e você tem o número que muda decisão de adoção.


## Quando o cache vale e quando não vale

**Vale muito:**
- Prompts com prefixo grande e estável (constituição, política, exemplos few-shot longos)
- Conversas multi-turno com o mesmo agente
- RAG com documentos carregados que se repetem entre consultas
- Agentes de codificação que mantêm o mesmo system prompt ao longo da sessão

**Não vale:**
- Prompts pequenos (menores que ~1.000 tokens)
- Quando o prefixo muda a cada chamada (sem prefixo estável, não há o que cachear)
- Quando a frequência de uso é tão baixa que o cache expira (5 min default) antes de ser reusado

**Regra de bolso:** se o seu agente tem prefixo fixo > 1.024 tokens e é chamado pelo menos 2x em 5 minutos, ative caching. É a otimização com maior ROI por hora de engenharia que existe em LLM hoje.


## O que fazer com isso

1. **Audite hoje os 3 agentes que mais consomem tokens na sua organização.** Se algum tem prefixo fixo > 1.024 tokens e ainda não usa caching, esse é seu próximo PR.
2. **No próximo desenho de prompt, separe explicitamente prefixo estável e parte variável.** Coloque o estável primeiro, marque com `cache_control`.
3. **No Apêndice J do livro, confira o tarifário corrente** — preços de cache e janelas de TTL (5 min vs 1h) mudam.

**Próximo passo:** este foi o último notebook fundacional. O próximo nível é construir RAG completo (planejado para release futuro). Por enquanto, volte ao livro: Capítulo 18 cobre economia de tokens em profundidade, com o Framework F7 (Custo Composto em Três Tempos) como instrumento de decisão.
